# Pistas de audio, mezcla temporal y Transformada de Fourier

Curso: **ACUS099 - Procesamiento Digital de Señales** con Python  
Tema: **Audio multicanal/multipista, suma de señales y análisis espectral con FFT**

En esta clase trabajaremos con pistas de audio almacenadas localmente en:

```text
../data/audio_tracks/
```

Las pistas esperadas son:

```text
 bass.wav
 drums.wav
 piano.wav
 voice1.wav
 voice2.wav
```

> Nota importante: los archivos de audio no se suben al repositorio por razones de derecho de autor. Cada estudiante debe trabajar con sus archivos locales.

## Objetivos de la clase

Al finalizar esta actividad, deberías poder:

1. Cargar pistas de audio en Python.
2. Visualizar señales en el dominio temporal.
3. Calcular duración, número de muestras y frecuencia de muestreo.
4. Resolver el problema de pistas con distinta duración.
5. Sumar señales para construir una mezcla simple.
6. Aplicar la Transformada Discreta de Fourier usando `np.fft.fft`.
7. Relacionar la formulación matricial de la DFT:

$
X = W x
$

con el uso práctico del algoritmo FFT.

## 1. Importar librerías

Usaremos:

- `numpy` para cálculo numérico.
- `matplotlib` para gráficos.
- `librosa` para cargar audio.
- `soundfile` para guardar audio.
- `pathlib` para trabajar con rutas de archivos.

In [5]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.signal import welch
from IPython.display import Audio, display


## 2. Definir carpeta y nombres de las pistas

La carpeta de trabajo del notebook será `notebooks/`, por eso usamos `../data/audio_tracks/` para subir un nivel y entrar a la carpeta `data`.

In [6]:
audio_dir = Path("../data/audio_tracks")

track_files = {
    "bass": audio_dir / "bass.wav",
    "drums": audio_dir / "drums.wav",
    "piano": audio_dir / "piano.wav",
    "voice1": audio_dir / "voice1.wav",
    "voice2": audio_dir / "voice2.wav",
}

track_files

{'bass': WindowsPath('../data/audio_tracks/bass.wav'),
 'drums': WindowsPath('../data/audio_tracks/drums.wav'),
 'piano': WindowsPath('../data/audio_tracks/piano.wav'),
 'voice1': WindowsPath('../data/audio_tracks/voice1.wav'),
 'voice2': WindowsPath('../data/audio_tracks/voice2.wav')}

## 3. Verificar que los archivos existen

Antes de cargar los audios, revisamos si Python encuentra los archivos.

In [7]:
for name, path in track_files.items():
    if path.exists():
        print(f"{name:8s}: OK -> {path}")
    else:
        print(f"{name:8s}: NO encontrado -> {path}")

bass    : NO encontrado -> ..\data\audio_tracks\bass.wav
drums   : NO encontrado -> ..\data\audio_tracks\drums.wav
piano   : NO encontrado -> ..\data\audio_tracks\piano.wav
voice1  : NO encontrado -> ..\data\audio_tracks\voice1.wav
voice2  : NO encontrado -> ..\data\audio_tracks\voice2.wav


## 4. Cargar las pistas con una frecuencia de muestreo común

Para poder sumar señales de audio, todas deben tener la misma frecuencia de muestreo. En esta clase usaremos:

$
f_s = 44100 \; \text{Hz}
$

Esto significa que cada segundo de audio se representa con 44100 muestras.

In [8]:
target_sr = 44100

tracks = {}

for name, path in track_files.items():
    signal, sr = librosa.load(path, sr=target_sr, mono=True)
    tracks[name] = signal

print("Pistas cargadas correctamente.\n")

for name, signal in tracks.items():
    duration = len(signal) / target_sr
    print(f"{name:8s} | samples = {len(signal):9d} | duration = {duration:8.2f} s")

C:\Users\tovig\AppData\Local\Temp\ipykernel_5468\1345837939.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  signal, sr = librosa.load(path, sr=target_sr, mono=True)


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\audio_tracks\\bass.wav'

## 4.1 Escuchar las pistas dentro del cuadernillo

Además de mirar las formas de onda, podemos escuchar las señales directamente en el notebook usando `IPython.display.Audio`.

Esto es muy útil para conectar lo que vemos en el dominio temporal con lo que percibimos auditivamente.

> Nota: si las pistas son largas, en clase podemos escuchar solo algunos segundos usando un recorte temporal.


In [ ]:
# Escuchar algunos segundos de cada pista
start_time = 0      # seconds
duration_time = 10  # seconds

start_sample = int(start_time * target_sr)
end_sample = int((start_time + duration_time) * target_sr)

for name, signal in tracks.items():
    print(f'Listening to: {name}')
    display(Audio(signal[start_sample:end_sample], rate=target_sr))


## 5. Visualizar una pista en el dominio temporal

Una señal de audio digital es una secuencia de muestras. Si la frecuencia de muestreo es `target_sr`, entonces el eje temporal se construye como:

$
t[n] = \frac{n}{f_s}
$

In [ ]:
track_name = "voice1"
x = tracks[track_name]

t = np.arange(len(x)) / target_sr

plt.figure(figsize=(12, 4))
plt.plot(t, x)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title(f"Waveform: {track_name}")
plt.grid(True)
plt.show()

## 6. Visualizar todas las pistas

Ahora graficamos todas las pistas para comparar sus formas de onda.

In [ ]:
plt.figure(figsize=(12, 9))

for i, (name, signal) in enumerate(tracks.items()):
    t = np.arange(len(signal)) / target_sr
    
    plt.subplot(len(tracks), 1, i + 1)
    plt.plot(t, signal)
    plt.title(name)
    plt.ylabel("Amplitude")
    plt.grid(True)

plt.xlabel("Time [s]")
plt.tight_layout()
plt.show()

## 7. Comparar duraciones

Para sumar señales muestra a muestra, todas las señales deben tener el mismo número de muestras.

Si las pistas tienen distinta duración, entonces sus vectores tienen distinto largo y no se pueden sumar directamente.

In [ ]:
durations = {name: len(signal) / target_sr for name, signal in tracks.items()}

plt.figure(figsize=(8, 4))
plt.bar(durations.keys(), durations.values())
plt.ylabel("Duration [s]")
plt.title("Duration of each audio track")
plt.grid(axis="y")
plt.show()

In [ ]:
lengths = {name: len(signal) for name, signal in tracks.items()}

min_length = min(lengths.values())
max_length = max(lengths.values())

print("Shortest track:", min_length, "samples")
print("Longest track :", max_length, "samples")
print("Shortest duration:", min_length / target_sr, "s")
print("Longest duration :", max_length / target_sr, "s")

## 8. Estrategia 1: recortar todas las pistas al largo mínimo

Esta estrategia conserva solo el tramo común de todas las pistas. Es simple y evita agregar silencio artificial.

In [ ]:
tracks_trimmed = {}

for name, signal in tracks.items():
    tracks_trimmed[name] = signal[:min_length]

print("Todas las pistas recortadas a", min_length, "muestras.")

## 9. Estrategia 2: rellenar con ceros hasta el largo máximo

Esta estrategia conserva la duración máxima. Las pistas más cortas se completan con silencio al final.

In [ ]:
tracks_padded = {}

for name, signal in tracks.items():
    padded = np.zeros(max_length)
    padded[:len(signal)] = signal
    tracks_padded[name] = padded

print("Todas las pistas rellenadas a", max_length, "muestras.")

## 10. Estrategia 1: mezclar recortando todas las pistas al largo mínimo

La suma se realiza muestra a muestra:

$
y[n] = x_1[n] + x_2[n] + x_3[n] + \cdots
$

Después normalizamos para evitar saturación.

## Analogía con Excel:

Imaginen que cada pista es una columna de números en una planilla Excel.

En la fila 1 tenemos la amplitud de todas las pistas en el primer instante.
En la fila 2, la amplitud en el segundo instante.

Mezclar el audio es sumar cada fila. Eso es exactamente lo que hace Python, pero con miles o millones de muestras.

<img src="imagen_excel.png" width="500">

In [ ]:
bass_example = np.array([0.1, 0.2, 0.1, 0.0])
drums_example = np.array([0.0, 0.3, 0.0, 0.2])
piano_example = np.array([0.2, 0.1, 0.0, 0.1])

mix_example = bass_example + drums_example + piano_example

print("Bass :", bass_example)
print("Drums:", drums_example)
print("Piano:", piano_example)
print("Mix  :", mix_example)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(bass_example, marker="o", label="bass")
plt.plot(drums_example, marker="o", label="drums")
plt.plot(piano_example, marker="o", label="piano")
plt.plot(mix_example, marker="o", label="mix", linewidth=3)

plt.xlabel("Sample index n")
plt.ylabel("Amplitude")
plt.title("Example of sample-by-sample audio mixing")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Esto crea una señal vacía, llena de ceros, con la misma duración que las pistas recortadas.
# Ese vector "mix_trimmed" será la mezcla final.
mix_trimmed = np.zeros(min_length)

# Esto recorre una por una las pistas guardadas en el diccionario tracks_trimmed:
# bass, drums, piano, voice1, voice2.
# En cada vuelta del ciclo:
# * name contiene el nombre de la pista.
# * signal contiene el vector de audio de esa pista.
for name, signal in tracks_trimmed.items():
    # Esta línea significa: mix_trimmed = mix_trimmed + signal
    # Es decir, va acumulando las pistas en la mezcla.
    # Primero: mix_trimmed = zeros + bass
    # Después: mix_trimmed = bass + drums
    # Luego: mix_trimmed   = bass + drums + piano
    # Después: mix_trimmed = bass + drums + piano + voice1
    # Y luego: mix_trimmed = bass + drums + piano + voice1 + voice2
    mix_trimmed += signal

# Normalización
mix_trimmed = mix_trimmed / np.max(np.abs(mix_trimmed))

print("Mezcla recortada creada.")
print("Samples:", len(mix_trimmed))
print("Duration:", len(mix_trimmed) / target_sr, "s")

In [ ]:
t = np.arange(len(mix_trimmed)) / target_sr

plt.figure(figsize=(12, 4))
plt.plot(t, mix_trimmed)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title("Mixed signal using trimmed tracks")
plt.grid(True)
plt.show()

## 11. Estrategia 2: mezclar rellenando con ceros hasta el largo máximo

Esta versión tendrá la duración de la pista más larga.

In [ ]:
mix_padded = np.zeros(max_length)

for name, signal in tracks_padded.items():
    mix_padded += signal

mix_padded = mix_padded / np.max(np.abs(mix_padded))

print("Mezcla con zero-padding creada.")
print("Samples:", len(mix_padded))
print("Duration:", len(mix_padded) / target_sr, "s")

In [ ]:
t = np.arange(len(mix_padded)) / target_sr

plt.figure(figsize=(12, 4))
plt.plot(t, mix_padded)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title("Mixed signal using zero-padded tracks")
plt.grid(True)
plt.show()

## 12. Guardar las mezclas

Guardamos ambas mezclas en la carpeta `outputs/audio/`.

In [ ]:
output_audio_dir = Path("../outputs/audio")
output_audio_dir.mkdir(parents=True, exist_ok=True)

sf.write(output_audio_dir / "mix_trimmed.wav", mix_trimmed, target_sr)
sf.write(output_audio_dir / "mix_padded.wav", mix_padded, target_sr)

print("Archivos guardados en:", output_audio_dir)

## 12.1 Escuchar las mezclas generadas

Ahora podemos escuchar las dos mezclas creadas. Esto permite comparar auditivamente la diferencia entre recortar al largo mínimo y rellenar con ceros hasta el largo máximo.


In [ ]:
print('Mix using trimmed tracks')
display(Audio(mix_trimmed, rate=target_sr))

print('Mix using zero-padded tracks')
display(Audio(mix_padded, rate=target_sr))


## 13. Recordatorio: DFT como multiplicación matricial

La Transformada Discreta de Fourier puede escribirse como:

\[
X = W x
\]

donde:

- \(x\) es la señal en el tiempo.
- \(W\) es la matriz de Fourier.
- \(X\) es la señal transformada al dominio de la frecuencia.

Los elementos de la matriz \(W\) son:

\[
W_{k,n} = e^{-j 2\pi kn/N}
\]

donde:

- \(k\) es el índice de frecuencia.
- \(n\) es el índice de tiempo discreto.
- \(N\) es el número de muestras.

In [ ]:
x_small = np.array([1, 2, 3, 4], dtype=float)

N = len(x_small)
n = np.arange(N)
k = n.reshape((N, 1))

W = np.exp(-2j * np.pi * k * n / N)

X_matrix = W @ x_small
X_fft = np.fft.fft(x_small)

print("x:")
print(x_small)

print("\nDFT usando matriz W:")
print(X_matrix)

print("\nDFT usando np.fft.fft:")
print(X_fft)

print("\n¿Son equivalentes?")
print(np.allclose(X_matrix, X_fft))

## 14. Visualizar la matriz de Fourier

La matriz \(W\) tiene valores complejos. Podemos visualizar, por ejemplo, su parte real.

In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(W.real, aspect="auto")
plt.colorbar(label="Real value")
plt.xlabel("n: discrete time index")
plt.ylabel("k: frequency index")
plt.title("Real part of the Fourier matrix W")
plt.show()

## 15. Aplicar FFT a una pista individual

Ahora aplicamos la FFT a una pista real de audio. Como la señal es real, visualizaremos solo las frecuencias positivas.

In [ ]:
track_name = "voice1"
x_audio = tracks_trimmed[track_name]

X = np.fft.fft(x_audio)
freqs = np.fft.fftfreq(len(x_audio), d=1/target_sr)

N = len(x_audio)
positive_freqs = freqs[:N // 2]
positive_mag = np.abs(X[:N // 2])

plt.figure(figsize=(12, 4))
plt.plot(positive_freqs, positive_mag)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.title(f"FFT magnitude spectrum: {track_name}")
plt.grid(True)
plt.xlim(0, 10000)
plt.show()

## 16. Magnitud en decibeles

Para señales de audio, muchas veces es más informativo graficar la magnitud en escala logarítmica:

\[
20 \log_{10}(|X[k]|)
\]

In [ ]:
positive_mag_db = 20 * np.log10(positive_mag + 1e-12)

plt.figure(figsize=(12, 4))
plt.plot(positive_freqs, positive_mag_db)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title(f"FFT magnitude spectrum: {track_name}")
plt.grid(True)
plt.xlim(0, 10000)
plt.show()

## 17. Aplicar FFT a la mezcla

Ahora analizamos el contenido frecuencial de la mezcla.

In [ ]:
X_mix = np.fft.fft(mix_trimmed)
freqs_mix = np.fft.fftfreq(len(mix_trimmed), d=1/target_sr)

N = len(mix_trimmed)
positive_freqs_mix = freqs_mix[:N // 2]
positive_mag_mix = np.abs(X_mix[:N // 2])
positive_mag_mix_db = 20 * np.log10(positive_mag_mix + 1e-12)

plt.figure(figsize=(12, 4))
plt.plot(positive_freqs_mix, positive_mag_mix_db)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("FFT magnitude spectrum: mixed signal")
plt.grid(True)
plt.xlim(0, 10000)
plt.show()

## 18. Comparar espectros de todas las pistas

Aquí podemos comparar qué pistas tienen más energía relativa en distintas zonas del espectro.

In [ ]:
plt.figure(figsize=(12, 6))

for name, signal in tracks_trimmed.items():
    X = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), d=1/target_sr)
    
    positive_freqs = freqs[:len(signal) // 2]
    positive_mag = np.abs(X[:len(signal) // 2])
    positive_mag_db = 20 * np.log10(positive_mag + 1e-12)
    
    plt.plot(positive_freqs, positive_mag_db, label=name, alpha=0.8)

plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude [dB]")
plt.title("Frequency spectra of audio tracks")
plt.grid(True)
plt.xlim(0, 10000)
plt.legend()
plt.show()

## 18.1 Comparar PSD de todas las pistas y observar envolventes suaves

La FFT anterior muestra el espectro completo de cada señal, pero puede verse muy irregular porque depende de todos los detalles temporales de la pista.

Una alternativa muy usada en procesamiento de señales es estimar la **densidad espectral de potencia** o **PSD** (*Power Spectral Density*). Aquí usaremos el método de Welch, que divide la señal en ventanas, calcula espectros locales y luego los promedia.

Esto produce curvas más suaves, útiles para observar la envolvente espectral general de cada pista.

> Por ahora usaremos la PSD como herramienta exploratoria, sin entrar todavía en todos los detalles matemáticos del método.


In [ ]:
plt.figure(figsize=(12, 6))

nperseg = 4096

for name, signal in tracks_trimmed.items():
    freqs_psd, psd = welch(
        signal,
        fs=target_sr,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        window='hann'
    )
    
    psd_db = 10 * np.log10(psd + 1e-20)
    plt.plot(freqs_psd, psd_db, label=name, alpha=0.9)

plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [dB/Hz]')
plt.title('Power Spectral Density of audio tracks - Welch method')
plt.grid(True)
plt.xlim(0, 10000)
plt.legend()
plt.show()


También podemos comparar la PSD de la mezcla con las PSD de las pistas individuales.


In [ ]:
plt.figure(figsize=(12, 6))

for name, signal in tracks_trimmed.items():
    freqs_psd, psd = welch(signal, fs=target_sr, nperseg=4096, noverlap=2048, window='hann')
    psd_db = 10 * np.log10(psd + 1e-20)
    plt.plot(freqs_psd, psd_db, label=name, alpha=0.5)

freqs_mix_psd, psd_mix = welch(mix_trimmed, fs=target_sr, nperseg=4096, noverlap=2048, window='hann')
psd_mix_db = 10 * np.log10(psd_mix + 1e-20)

plt.plot(freqs_mix_psd, psd_mix_db, label='mix_trimmed', linewidth=2.5)

plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [dB/Hz]')
plt.title('PSD comparison: individual tracks and mixed signal')
plt.grid(True)
plt.xlim(0, 10000)
plt.legend()
plt.show()


## 19. Actividad guiada

Responde en tu cuaderno o en una celda Markdown:

1. ¿Todas las pistas tienen la misma duración?
2. ¿Por qué no podemos sumar directamente señales con distinto número de muestras?
3. ¿Qué diferencia conceptual hay entre recortar y rellenar con ceros?
4. ¿Por qué normalizamos después de sumar las pistas?
5. ¿Qué representa el eje de frecuencia en los gráficos de FFT?
6. ¿Por qué la expresión \(X = Wx\) es útil para entender la DFT, pero poco práctica para señales largas?
7. ¿Qué diferencia observas entre el espectro de una pista individual y el espectro de la mezcla?
8. ¿Qué diferencia visual observas entre el espectro calculado directamente con FFT y la PSD estimada con Welch?
9. ¿Qué pista parece concentrar más energía en bajas frecuencias? ¿Cuál parece tener más energía en frecuencias medias o altas?


## 20. Desafío opcional

Modifica el notebook para:

1. Calcular la FFT solo de los primeros 10 segundos de cada pista.
2. Comparar los espectros entre 0 y 2000 Hz.
3. Guardar una figura en `outputs/figures/`.
4. Crear una mezcla solo con `bass`, `drums` y `piano`.
5. Crear otra mezcla solo con `voice1` y `voice2`.

In [ ]:
# Celda libre para el desafío opcional